# 03 — Control Policy Comparison

**Repository:** `quantum-hardware-company`  
**Directory:** `control-stack/`

This notebook benchmarks multiple drift-control policies using the same synthetic calibration environment as notebooks 01–02.

## Purpose

Notebook 01 showed that feedback compensation works.  
Notebook 02 showed that Kalman filtering outperforms moving-average smoothing.

This notebook asks:

> Which control policy best stabilizes response-level quantum behavior under drift?

## Policies

- Policy 0 — no compensation
- Policy 1 — moving-average compensation
- Policy 2 — Kalman filtered compensation
- Policy 3 — one-step predictive Kalman compensation
- Policy 4 — oracle compensation baseline

## Principle

A control policy should be judged by observable physics:

- parameter error
- response-level error
- worst-case stability
- CGCS / Triplet-Phase-Lock margin


## Control objective

For a target Rabi-like response,

$$
P(t)=A\sin^2\left(\frac{\Omega t}{2}\right)+B
$$

drift changes effective parameters:

$$
\Omega_k = \Omega_{target} + \delta\Omega_k
$$

$$
B_k = B_{target} + \delta B_k
$$

A control policy estimates drift and applies compensation.

The best policy minimizes response-level deviation:

$$
\mathrm{RMSE}\left(P_{controlled}(t),P_{target}(t)\right)
$$

while maintaining CGCS stability:

$$
\cos\theta \geq \frac{1}{\sqrt{1^2+1^2}}
$$


In [ ]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures"
RESULTS_DIR = "results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
def rabi_model(t, A, Omega, B):
    """Rabi oscillation probability model."""
    return A * np.sin(0.5 * Omega * t) ** 2 + B


def fit_model(model, x, y, p0, bounds=(-np.inf, np.inf)):
    """Fit model to data and return best-fit params + covariance."""
    params, cov = curve_fit(model, x, y, p0=p0, bounds=bounds, maxfev=30000)
    return params, cov


def phase_lock_metric(signal, target):
    """Cosine similarity between signal and target."""
    dot = np.dot(signal, target)
    norm = np.linalg.norm(signal) * np.linalg.norm(target)
    if norm == 0:
        return np.nan
    return dot / norm


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def moving_average(x, window=4):
    """Causal moving average estimate."""
    y = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        y[i] = np.mean(x[lo:i + 1])
    return y


def kalman_filter_1d(z, q, r, x0=None, p0=1.0):
    """Simple 1D random-walk Kalman filter.

    State:
        x_k = x_{k-1} + w_k
        z_k = x_k + v_k
    """
    z = np.asarray(z, dtype=float)
    x_hat = np.zeros_like(z)
    x_pred_hist = np.zeros_like(z)
    p_hist = np.zeros_like(z)
    k_hist = np.zeros_like(z)

    x = z[0] if x0 is None else float(x0)
    p = float(p0)

    for i, measurement in enumerate(z):
        # Predict
        x_pred = x
        p_pred = p + q

        # Update
        k_gain = p_pred / (p_pred + r)
        x = x_pred + k_gain * (measurement - x_pred)
        p = (1 - k_gain) * p_pred

        x_pred_hist[i] = x_pred
        x_hat[i] = x
        p_hist[i] = p
        k_hist[i] = k_gain

    return x_hat, x_pred_hist, p_hist, k_hist


def one_step_prediction_from_filtered(x_hat):
    """One-step prediction for a random-walk model.

    For x_k = x_{k-1} + w_k, the one-step-ahead prediction is the current filtered state.
    We use shifted filtered state to represent the value available before measuring the next block.
    """
    pred = np.zeros_like(x_hat)
    pred[0] = x_hat[0]
    pred[1:] = x_hat[:-1]
    return pred


def evaluate_policy(name, delta_omega_hat, delta_b_hat, true_delta_omega, true_delta_b):
    """Apply compensation estimates and evaluate parameter, response, and CGCS metrics."""
    omega_controlled = target_Omega + true_delta_omega - delta_omega_hat
    b_controlled = np.clip(target_B + true_delta_b - delta_b_hat, 0, 0.3)

    omega_uncompensated = target_Omega + true_delta_omega
    b_uncompensated = target_B + true_delta_b

    target_signal_local = rabi_model(pulse_t, target_A, target_Omega, target_B)

    response_rmse = []
    phase_lock = []
    controlled_signals = []

    for k in range(len(true_delta_omega)):
        y = rabi_model(pulse_t, target_A, omega_controlled[k], b_controlled[k])
        controlled_signals.append(y)
        response_rmse.append(rmse(y, target_signal_local))
        phase_lock.append(phase_lock_metric(y, target_signal_local))

    response_rmse = np.array(response_rmse)
    phase_lock = np.array(phase_lock)
    controlled_signals = np.array(controlled_signals)

    return {
        "name": name,
        "omega_controlled": omega_controlled,
        "b_controlled": b_controlled,
        "omega_error": omega_controlled - target_Omega,
        "b_error": b_controlled - target_B,
        "omega_rmse": rmse(omega_controlled, np.full(len(omega_controlled), target_Omega)),
        "b_rmse": rmse(b_controlled, np.full(len(b_controlled), target_B)),
        "response_rmse": response_rmse,
        "response_rmse_mean": float(np.mean(response_rmse)),
        "response_rmse_max": float(np.max(response_rmse)),
        "phase_lock": phase_lock,
        "min_phase_lock": float(np.min(phase_lock)),
        "mean_phase_lock": float(np.mean(phase_lock)),
        "max_angle_degrees": float(np.max(np.degrees(np.arccos(np.clip(phase_lock, -1, 1))))),
        "controlled_signals": controlled_signals,
    }


## Generate synthetic drift environment

We use the same base environment as notebooks 01–02, then benchmark all policies on identical drift.


In [ ]:
# Calibration blocks / lab-time index
n_blocks = 36
block_times = np.arange(n_blocks)

# Pulse duration axis inside each calibration block
n_points_per_block = 140
pulse_t = np.linspace(0, 10, n_points_per_block)

# Target parameters
target_A = 0.88
target_Omega = 2.45
target_B = 0.06

target_signal = rabi_model(pulse_t, target_A, target_Omega, target_B)

# True environmental drift
true_delta_Omega = (
    0.055 * np.sin(2 * np.pi * block_times / 21 + 0.4)
    + 0.018 * np.sin(2 * np.pi * block_times / 9)
)
true_delta_B = (
    0.0016 * block_times
    + 0.006 * np.sin(2 * np.pi * block_times / 14 + 0.2)
)

Omega_uncomp = target_Omega + true_delta_Omega
B_uncomp = target_B + true_delta_B
A_uncomp = target_A + 0.012 * np.sin(2 * np.pi * block_times / 17)

print(f"Target Omega = {target_Omega:.4f}")
print(f"Target B     = {target_B:.4f}")
print(f"Uncompensated Omega range = {Omega_uncomp.min():.4f} to {Omega_uncomp.max():.4f}")
print(f"Uncompensated B range     = {B_uncomp.min():.4f} to {B_uncomp.max():.4f}")


## Simulate calibration measurements

Each block is measured, then fit to recover estimated drift measurements.


In [ ]:
Y_obs = []
Y_clean = []

for k in range(n_blocks):
    y_clean = rabi_model(pulse_t, A_uncomp[k], Omega_uncomp[k], B_uncomp[k])
    noise = 0.04 * np.random.randn(n_points_per_block)
    y_obs = np.clip(y_clean + noise, 0, 1)
    Y_clean.append(y_clean)
    Y_obs.append(y_obs)

Y_clean = np.array(Y_clean)
Y_obs = np.array(Y_obs)

fit_params = []
fit_stds = []

initial_guess = [0.85, 2.40, 0.05]
bounds = ([0.0, 0.0, 0.0], [1.2, 5.0, 0.3])

for k in range(n_blocks):
    params, cov = fit_model(
        rabi_model,
        pulse_t,
        Y_obs[k],
        p0=initial_guess,
        bounds=bounds,
    )
    fit_params.append(params)
    fit_stds.append(np.sqrt(np.diag(cov)))

fit_params = np.array(fit_params)
fit_stds = np.array(fit_stds)

Omega_est = fit_params[:, 1]
B_est = fit_params[:, 2]

delta_Omega_est = Omega_est - target_Omega
delta_B_est = B_est - target_B

print("Block-wise calibration fits complete.")
print(f"Mean estimated Omega drift = {np.mean(delta_Omega_est):.6f}")
print(f"Mean estimated B drift     = {np.mean(delta_B_est):.6f}")


## Define control policies

We compare:

- no compensation
- moving average
- filtered Kalman
- one-step predictive Kalman
- oracle baseline

The oracle sees true drift and represents the ideal compensation lower bound.


In [ ]:
# Policy 0: no compensation
delta_Omega_none = np.zeros(n_blocks)
delta_B_none = np.zeros(n_blocks)

# Policy 1: moving average
ma_window = 4
delta_Omega_ma = moving_average(delta_Omega_est, window=ma_window)
delta_B_ma = moving_average(delta_B_est, window=ma_window)

# Policy 2: Kalman filtered estimate
r_omega = float(np.median(fit_stds[:, 1] ** 2))
r_b = float(np.median(fit_stds[:, 2] ** 2))

q_omega = 8.0e-4
q_b = 1.0e-5

delta_Omega_kf, delta_Omega_pred_internal, p_Omega_hist, k_Omega_hist = kalman_filter_1d(
    delta_Omega_est,
    q=q_omega,
    r=r_omega,
    p0=1.0,
)

delta_B_kf, delta_B_pred_internal, p_B_hist, k_B_hist = kalman_filter_1d(
    delta_B_est,
    q=q_b,
    r=r_b,
    p0=1.0,
)

# Policy 3: one-step predictive Kalman estimate available before the next block measurement.
delta_Omega_pkf = one_step_prediction_from_filtered(delta_Omega_kf)
delta_B_pkf = one_step_prediction_from_filtered(delta_B_kf)

# Policy 4: oracle baseline
delta_Omega_oracle = true_delta_Omega.copy()
delta_B_oracle = true_delta_B.copy()

policies = {
    "none": (delta_Omega_none, delta_B_none),
    "moving_average": (delta_Omega_ma, delta_B_ma),
    "kalman": (delta_Omega_kf, delta_B_kf),
    "predictive_kalman": (delta_Omega_pkf, delta_B_pkf),
    "oracle": (delta_Omega_oracle, delta_B_oracle),
}

print("Policies defined:")
for name in policies:
    print(f"- {name}")


## Visualize policy estimates

Estimator plots show why each policy behaves differently.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(block_times, true_delta_Omega, linewidth=2, label="true drift")
plt.plot(block_times, delta_Omega_est, marker="o", linewidth=1, label="measured drift")
plt.plot(block_times, delta_Omega_ma, linewidth=2, linestyle="--", label="moving average")
plt.plot(block_times, delta_Omega_kf, linewidth=2, linestyle=":", label="Kalman")
plt.plot(block_times, delta_Omega_pkf, linewidth=2, linestyle="-.", label="predictive Kalman")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("Omega drift estimate")
plt.title("Policy estimator comparison: Rabi frequency")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/policy_01_omega_control_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/policy_01_omega_control_comparison.pdf")

plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(block_times, true_delta_B, linewidth=2, label="true drift")
plt.plot(block_times, delta_B_est, marker="o", linewidth=1, label="measured drift")
plt.plot(block_times, delta_B_ma, linewidth=2, linestyle="--", label="moving average")
plt.plot(block_times, delta_B_kf, linewidth=2, linestyle=":", label="Kalman")
plt.plot(block_times, delta_B_pkf, linewidth=2, linestyle="-.", label="predictive Kalman")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("offset drift estimate")
plt.title("Policy estimator comparison: readout offset")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/policy_02_offset_control_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/policy_02_offset_control_comparison.pdf")

plt.show()


## Evaluate policies

Each policy produces controlled parameters, response RMSE, and CGCS metrics.


In [ ]:
policy_results = {}

for name, (delta_omega_hat, delta_b_hat) in policies.items():
    policy_results[name] = evaluate_policy(
        name,
        delta_omega_hat,
        delta_b_hat,
        true_delta_Omega,
        true_delta_B,
    )

for name, result in policy_results.items():
    print(
        f"{name:18s} | "
        f"response RMSE mean = {result['response_rmse_mean']:.6f} | "
        f"max = {result['response_rmse_max']:.6f} | "
        f"min phase-lock = {result['min_phase_lock']:.6f}"
    )


## Response-level RMSE comparison

This is the primary control-policy benchmark.


In [ ]:
plt.figure(figsize=(10, 5))
for name, result in policy_results.items():
    plt.plot(block_times, result["response_rmse"], marker="o", linewidth=1, label=name)

plt.xlabel("calibration block / lab time index")
plt.ylabel("RMSE vs target response")
plt.title("Control policy comparison: response-level error")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/policy_03_response_rmse_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/policy_03_response_rmse_comparison.pdf")

plt.show()


## CGCS phase-lock stability comparison

All useful policies should remain above the 45° threshold.

A stronger policy improves margin toward cosine similarity 1.


In [ ]:
threshold = 1 / np.sqrt(2)

plt.figure(figsize=(10, 5))
plt.axhline(threshold, linewidth=1, linestyle="--", label="45° threshold")

for name, result in policy_results.items():
    plt.plot(block_times, result["phase_lock"], marker="o", linewidth=1, label=name)

plt.ylim(0.68, 1.01)
plt.xlabel("calibration block / lab time index")
plt.ylabel("cosine similarity vs target")
plt.title("Control policy comparison: CGCS phase-lock stability")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/policy_04_cgcs_stability_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/policy_04_cgcs_stability_comparison.pdf")

plt.show()


## Worst-case block comparison

We choose the block where no compensation has the largest response error.


In [ ]:
worst_block = int(np.argmax(policy_results["none"]["response_rmse"]))

plt.figure(figsize=(10, 5))
plt.plot(pulse_t, target_signal, linewidth=3, label="target")

for name, result in policy_results.items():
    plt.plot(pulse_t, result["controlled_signals"][worst_block], linewidth=2, label=name)

plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Worst-case block response comparison: block {worst_block}")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/policy_05_worst_case_block_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/policy_05_worst_case_block_comparison.pdf")

plt.show()


## Policy ranking table

Rank policies by mean response-level RMSE.

This table is the decision layer: which policy best preserves target behavior?


In [ ]:
ranking = sorted(
    policy_results.values(),
    key=lambda r: r["response_rmse_mean"],
)

policy_table = []
for rank, result in enumerate(ranking, start=1):
    policy_table.append({
        "rank": rank,
        "policy": result["name"],
        "omega_rmse": result["omega_rmse"],
        "b_rmse": result["b_rmse"],
        "response_rmse_mean": result["response_rmse_mean"],
        "response_rmse_max": result["response_rmse_max"],
        "min_phase_lock": result["min_phase_lock"],
        "mean_phase_lock": result["mean_phase_lock"],
        "max_angle_degrees": result["max_angle_degrees"],
        "all_blocks_phase_locked": bool(np.all(result["phase_lock"] >= threshold)),
    })

policy_table


## Policy ranking summary figure


In [ ]:
policy_names_ranked = [row["policy"] for row in policy_table]
response_means_ranked = [row["response_rmse_mean"] for row in policy_table]
response_max_ranked = [row["response_rmse_max"] for row in policy_table]

x = np.arange(len(policy_names_ranked))
width = 0.35

plt.figure(figsize=(10, 5))
bars_mean = plt.bar(x - width / 2, response_means_ranked, width, label="mean RMSE")
bars_max = plt.bar(x + width / 2, response_max_ranked, width, label="max RMSE")

plt.xticks(x, policy_names_ranked, rotation=25, ha="right")
plt.ylabel("response RMSE")
plt.title("Control policy ranking: response-level error")
plt.legend()

for bars in [bars_mean, bars_max]:
    for bar in bars:
        value = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.002,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

plt.tight_layout()

plt.savefig(f"{FIG_DIR}/policy_06_policy_ranking_summary.png", dpi=300)
plt.savefig(f"{FIG_DIR}/policy_06_policy_ranking_summary.pdf")

plt.show()


## Parameter RMSE summary

Parameter RMSE shows whether policies improve control coordinates, not just final response.


In [ ]:
metric_labels = ["Omega RMSE", "Offset RMSE"]
policy_names = list(policy_results.keys())

omega_values = [policy_results[name]["omega_rmse"] for name in policy_names]
b_values = [policy_results[name]["b_rmse"] for name in policy_names]

x = np.arange(len(policy_names))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - width / 2, omega_values, width, label="Omega RMSE")
plt.bar(x + width / 2, b_values, width, label="Offset RMSE")
plt.xticks(x, policy_names, rotation=25, ha="right")
plt.ylabel("parameter RMSE")
plt.title("Control policy comparison: parameter RMSE")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/policy_07_parameter_rmse_summary.png", dpi=300)
plt.savefig(f"{FIG_DIR}/policy_07_parameter_rmse_summary.pdf")

plt.show()


## Policy interpretation

Expected ranking:

```text
oracle > Kalman ≈ predictive Kalman > moving average > no compensation
```

In this random-walk Kalman setup, predictive Kalman uses the previous filtered state as the available pre-measurement estimate for the next block.

That means it can lag when drift changes quickly. A later notebook can improve this by adding a velocity state:

```text
x_k = [drift, drift_rate]
```


## Save policy comparison summary


In [ ]:
summary = {
    "n_blocks": int(n_blocks),
    "moving_average_window": int(ma_window),
    "kalman_q_omega": float(q_omega),
    "kalman_r_omega": float(r_omega),
    "kalman_q_b": float(q_b),
    "kalman_r_b": float(r_b),
    "phase_lock_threshold": float(threshold),
    "ranking": policy_table,
}

with open(f"{RESULTS_DIR}/control_policy_comparison_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/control_policy_comparison_summary.json")


## Zip outputs for download from Colab

Run this cell after all figures/results have been generated.


In [ ]:
!zip -r control_policy_comparison_outputs.zip figures results


## Control-stack status

This notebook adds the policy decision layer:

```text
01 drift compensation → 02 Kalman estimation → 03 policy comparison
```

## Result

The control stack can now evaluate policies by:

- parameter error
- response-level error
- worst-case response
- CGCS phase-lock margin

## Next steps

Possible next notebooks:

- `04_joint_parameter_filter.ipynb` — estimate Ω and B jointly with covariance.
- `05_velocity_state_kalman.ipynb` — add drift-rate state for predictive control.
- `noise-mitigation/01_structured_drift_model.ipynb` — model remaining residuals after control.

Guiding rule:

> Control policy is judged by stabilized physics, not by estimator aesthetics.
